# FGVC-Aircraft Classification — Final Report
A comprehensive report comparing deep learning architectures (ResNet-18 Frozen Baseline, ResNet-18 Unfrozen, ResNet-50, EfficientNet-B0, DINOv3), selecting the top-performing model, retraining on the full TrainVal dataset, and performing final evaluation on the unseen Test set.

### Google Colab Setup (If applicable)

In [ ]:
# Check if the notebook is run locally or on Google Colab instance
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point this to your project folder in Google Drive
    BASE_DIR = '/content/drive/MyDrive/project/aircraft_classification/'
else:
    # Local or standard Jupyter server
    BASE_DIR = './'

# Prepend BASE_DIR to sys.path
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# Dataset path
data_path = os.path.join(BASE_DIR, 'FGVCAircraft_Subset20')

In [ ]:
# Change working directory if running in Google Colab
if 'google.colab' in sys.modules and os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"Current working directory: {os.getcwd()}")

### Import essential libraries and configurations

In [ ]:
import os
import copy
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

%load_ext autoreload
%autoreload 2
from utils import set_seed, train_epoch

# Setup device & seed
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
set_seed(42)

In [ ]:
# ImageNet Statistics
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Validation / Test transform (deterministic)
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# Load test dataset
test_dataset = torchvision.datasets.ImageFolder(os.path.join(data_path, 'test'), transform=val_transform)
num_workers = 4 if torch.cuda.is_available() and os.name != 'nt' else 2
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)
class_names = test_dataset.classes
num_classes = len(class_names)

print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

### Model Sweep Comparison & Learning Curves Analysis

In `development_full.ipynb`, the following 5 models were systematically trained and evaluated under consistent hyperparameters:
* **Baseline:** ResNet-18 (Frozen Backbone) + AdamW + CosineAnnealingLR
* **H1:** ResNet-18 (Unfrozen Backbone) + AdamW + CosineAnnealingLR
* **H2:** ResNet-50 (Unfrozen Backbone) + AdamW + CosineAnnealingLR
* **H3:** EfficientNet-B0 (Unfrozen Backbone) + AdamW + CosineAnnealingLR
* **H4:** DINOv3 ViT-S/16 (Unfrozen / Feature Backbone) + AdamW + CosineAnnealingLR

In [ ]:
# Load the sweep results from development_full.ipynb
sweep_file = os.path.join(BASE_DIR, 'sweep_results.pkl') if os.path.exists(os.path.join(BASE_DIR, 'sweep_results.pkl')) else 'sweep_results.pkl'

try:
    with open(sweep_file, 'rb') as f:
        results = pickle.load(f)
    print("Successfully loaded sweep results.")
except FileNotFoundError:
    print(f"sweep_results.pkl not found at {sweep_file}! Please run development_full.ipynb first.")
    results = {}

if results:
    # 1. Comparative Analysis Table
    summary_data = []
    for name, res in results.items():
        summary_data.append({
            "Model Configuration": name,
            "Best Validation Accuracy": f"{res['best_val_acc']:.4f}",
            "Best Validation Accuracy (raw)": res["best_val_acc"]
        })

    df_summary = pd.DataFrame(summary_data).sort_values(by="Best Validation Accuracy (raw)", ascending=False).drop(columns=["Best Validation Accuracy (raw)"])
    print("\n" + "="*50)
    print("         MODEL PERFORMANCE COMPARISON")
    print("="*50)
    display(df_summary)

    # 2. Side-by-side plot for each model (Loss vs. Epochs and Accuracy vs. Epochs)
    for name, res in results.items():
        history = res["history"]
        epochs = range(1, len(history["train_loss"]) + 1)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        fig.suptitle(f"Learning Curves: {name}", fontsize=14, y=1.02)
        
        # Loss subplot
        axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker="o", linewidth=2)
        axes[0].plot(epochs, history["val_loss"], label="Validation Loss", marker="o", linewidth=2)
        axes[0].set_xlabel("Epochs")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Loss over Epochs")
        axes[0].legend()
        axes[0].grid(True, linestyle="--", alpha=0.7)
        
        # Accuracy subplot
        axes[1].plot(epochs, history["train_acc"], label="Train Accuracy", marker="x", linewidth=2)
        axes[1].plot(epochs, history["val_acc"], label="Validation Accuracy", marker="x", linewidth=2)
        axes[1].set_xlabel("Epochs")
        axes[1].set_ylabel("Balanced Accuracy")
        axes[1].set_title("Accuracy over Epochs")
        axes[1].legend()
        axes[1].grid(True, linestyle="--", alpha=0.7)
        
        plt.tight_layout()
        plt.show()

    # 3. Validation Accuracy comparison across all models over time
    plt.figure(figsize=(12, 6))
    for name, res in results.items():
        history = res["history"]
        epochs = range(1, len(history["val_acc"]) + 1)
        plt.plot(epochs, history["val_acc"], label=f"{name} (Best: {res['best_val_acc']:.3f})", marker="o", linewidth=2)

    plt.title("Validation Accuracy Across All Model Configurations", fontsize=14)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("Validation Accuracy (Balanced)", fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

### Retrain the Selected Best Model on Full TrainVal Dataset

In [ ]:
# Data augmentation for full retraining
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# Load the FULL trainval dataset (100% data, no split)
full_trainval_dataset = torchvision.datasets.ImageFolder(
    os.path.join(data_path, 'trainval'), transform=train_transform
)
full_train_loader = torch.utils.data.DataLoader(
    full_trainval_dataset, batch_size=16, shuffle=True, num_workers=num_workers
)
print(f"Loaded full TrainVal dataset: {len(full_trainval_dataset)} images.")

In [ ]:
def get_model(model_name, num_classes, freeze_backbone=False):
    """
    Initializes model architecture and adapts classification head.
    """
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
    elif model_name == 'dinov3':
        class DinoV3Classifier(nn.Module):
            def __init__(self, repo_dir, weights, num_classes, freeze_backbone=False):
                super().__init__()
                self.backbone = torch.hub.load(
                    repo_dir,
                    'dinov3_vits16',
                    source='local',
                    weights=weights
                )
                embed_dim = getattr(self.backbone, 'embed_dim', 384)
                self.fc = nn.Linear(embed_dim, num_classes)
                
                if freeze_backbone:
                    for param in self.backbone.parameters():
                        param.requires_grad = False

            def forward(self, x):
                features = self.backbone(x)
                return self.fc(features)
        
        base = BASE_DIR if 'BASE_DIR' in globals() else './'
        repo_dir = os.path.join(base, 'dinov3') if os.path.exists(os.path.join(base, 'dinov3')) else './dinov3'
        weights_path = os.path.join(base, 'weights', 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth') if os.path.exists(os.path.join(base, 'weights', 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth')) else './weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
        
        model = DinoV3Classifier(
            repo_dir=repo_dir,
            weights=weights_path,
            num_classes=num_classes,
            freeze_backbone=freeze_backbone
        )
    else:
        raise ValueError(f"Model {model_name} not supported")
    return model

In [ ]:
# Select the best model (default to 'dinov3' or top model from sweep)
best_model_name = 'dinov3'
freeze_backbone = False

print(f"Initializing best model architecture: {best_model_name} (freeze_backbone={freeze_backbone})")
final_model = get_model(best_model_name, num_classes=num_classes, freeze_backbone=freeze_backbone).to(device)

In [ ]:
# Retrain on full trainval dataset
lr = 0.001
total_epochs = 32

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()), lr=lr, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

patience = 5
epochs_no_improve = 0
best_train_loss = float("inf")
retrain_history = {"train_loss": [], "train_acc": []}

epoch_pbar = tqdm.tqdm(range(total_epochs), colour='blue', desc="Full Retraining")
for epoch in epoch_pbar:
    mean_train_loss, train_accuracy = train_epoch(
        final_model, full_train_loader, criterion, optimizer, epoch, device
    )
    scheduler.step()
    retrain_history["train_loss"].append(mean_train_loss)
    retrain_history["train_acc"].append(train_accuracy)
    
    epoch_pbar.write(f"Epoch [{epoch+1:2d}/{total_epochs:2d}] | Train Loss: {mean_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
    
    if mean_train_loss < best_train_loss:
        best_train_loss = mean_train_loss
        epochs_no_improve = 0
        torch.save(final_model.state_dict(), f"weights_final_{best_model_name}_full.pth")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs!")
            break

print("Full Retraining Complete! Saved weights to:", f"weights_final_{best_model_name}_full.pth")

### Final Evaluation on Unseen Test Set

In [ ]:
def evaluate_on_test(model, dataloader, device):
    """
    Evaluates model on the test dataset and prints classification report & balanced accuracy.
    """
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm.tqdm(dataloader, desc="Evaluating on Test Set"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            predicted = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    test_accuracy = balanced_accuracy_score(all_labels, all_preds)
    print(f"\n{'='*60}")
    print(f"Average Per-Class (Balanced) Accuracy on Test Set: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"{'='*60}\n")
    
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    return all_labels, all_preds, all_probs, test_accuracy

all_labels, all_preds, all_probs, test_accuracy = evaluate_on_test(final_model, test_loader, device)

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Compute per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)

# 1. Plot Per-Class Accuracy Bar Chart
plt.figure(figsize=(14, 6))
bars = plt.bar(class_names, per_class_acc, color='cornflowerblue', edgecolor='black', alpha=0.85)
plt.xlabel('Aircraft Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy on FGVC-Aircraft Test Set', fontsize=14)
plt.xticks(rotation=60, ha='right')
plt.ylim(0, 1.15)
plt.axhline(y=test_accuracy, color='crimson', linestyle='--', linewidth=2, label=f'Mean Balanced Accuracy: {test_accuracy:.4f}')

for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2., acc + 0.02, f"{acc:.2f}", ha='center', va='bottom', fontsize=9)

plt.legend(loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2. Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(11, 11))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=60, colorbar=True)
ax.grid(False)
plt.title('Confusion Matrix on Test Set', fontsize=14)
plt.tight_layout()
plt.show()